# RVL-CDIP Document Classifier — Production Training Notebook

Fine-tunes **ConvNeXt Tiny** on the RVL-CDIP 16-class document layout dataset.  
Produces the four artifacts that ship to the production repo:

| Artifact | Path in repo |
|---|---|
| Trained weights | `app/classifier/models/classifier.pt` |
| Model card + SHA-256 | `app/classifier/models/model_card.json` |
| 50 golden TIFFs | `app/classifier/eval/golden_images/` |
| Expected outputs | `app/classifier/eval/golden_expected.json` |

**Runtime:** Google Colab T4 GPU (free tier). Full training ≈ 90–120 min.  
**Run cells top-to-bottom in order. Do not skip cells.**

## Cell 1 — Install & imports

Pin versions to match `requirements.txt` in the production repo.

In [ ]:
# ── 1. Install extra packages not present in Colab by default ─────────────────
!pip install -q \
    torchmetrics==1.4.0 \
    matplotlib==3.9.0 \
    seaborn==0.13.2 \
    Pillow==10.4.0

# ── 2. Standard library ───────────────────────────────────────────────────────
import hashlib
import json
import os
import random
import shutil
import time
from pathlib import Path
from typing import Dict, List, Tuple

# ── 3. Numerics / vision ──────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image

# ── 4. Metrics & plotting ─────────────────────────────────────────────────────
from torchmetrics import Accuracy
from torchmetrics.classification import MulticlassConfusionMatrix
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── 5. Verify GPU ─────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch   : {torch.__version__}")
print(f"device  : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Training will be very slow.")

## Cell 2 — Global config

One place for every hyperparameter. Change values here only.

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Deterministic ops (small speed cost — required for golden set replay)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Dataset ───────────────────────────────────────────────────────────────────
# RVL-CDIP root after extracting the archive in Colab
DATA_ROOT = Path("/content/rvl-cdip")
LABELS_DIR = DATA_ROOT / "labels"   # train.txt / val.txt / test.txt
IMAGES_DIR = DATA_ROOT / "images"   # images/xxx/yyy/zzz.tif

CLASSES = [
    "letter", "form", "email", "handwritten", "advertisement",
    "scientific_report", "scientific_publication", "specification",
    "file_folder", "news_article", "budget", "invoice",
    "presentation", "questionnaire", "resume", "memo",
]
NUM_CLASSES = len(CLASSES)  # 16

# ── Model ─────────────────────────────────────────────────────────────────────
BACKBONE      = "convnext_tiny"          # or "convnext_small" for +1% accuracy at 2× cost
WEIGHTS_ENUM  = "IMAGENET1K_V1"          # exact torchvision weights identifier
IMG_SIZE      = 384                      # px — keeps fine document structure visible
CROP_SIZE     = 384                      # center-crop to same size

# ── Training schedule ─────────────────────────────────────────────────────────
BATCH_SIZE       = 32    # reduce to 16 if OOM on free Colab
NUM_WORKERS      = 2

PROBE_EPOCHS     = 5     # frozen backbone — head-only training
FINETUNE_EPOCHS  = 15    # partial unfreeze — backbone stages 6+7 + head
TOTAL_EPOCHS     = PROBE_EPOCHS + FINETUNE_EPOCHS

PROBE_LR         = 1e-3  # head during linear probe
HEAD_LR          = 5e-4  # head during fine-tune
BACKBONE_LR      = 1e-4  # unfrozen backbone stages during fine-tune

LABEL_SMOOTHING  = 0.1   # makes confidence honest
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 1.0
EARLY_STOP_PAT   = 4     # val accuracy patience

# ── Calibration & thresholds ──────────────────────────────────────────────────
TEMP_INIT        = 1.5   # starting point for temperature scaling search
REVIEWER_THRESH  = 0.7   # predictions below this are relabel-eligible

# ── Output paths (mirrored to Google Drive) ───────────────────────────────────
DRIVE_ROOT   = Path("/content/drive/MyDrive/rvlcdip_artifacts")
OUT_WEIGHTS  = DRIVE_ROOT / "classifier.pt"
OUT_CARD     = DRIVE_ROOT / "model_card.json"
OUT_GOLDEN   = DRIVE_ROOT / "golden_images"
OUT_EXPECTED = DRIVE_ROOT / "golden_expected.json"

GOLDEN_N     = 50        # total golden images
GOLDEN_EASY  = 2         # top-confidence picks per class  (2 × 16 = 32)
GOLDEN_HARD  = 18        # ambiguous cases picked from confusion matrix

# ── Accuracy threshold that production startup enforces ───────────────────────
MIN_TOP1_THRESHOLD = 0.90   # commit this value in README too

print("Config loaded.")
print(f"  backbone : {BACKBONE} / {WEIGHTS_ENUM}")
print(f"  img size : {IMG_SIZE}px")
print(f"  epochs   : {PROBE_EPOCHS} probe + {FINETUNE_EPOCHS} finetune = {TOTAL_EPOCHS}")
print(f"  batch    : {BATCH_SIZE}")
print(f"  device   : {DEVICE}")

## Cell 3 — Download & mount dataset

RVL-CDIP lives at adamharley.com. Download once and cache in Drive.

In [ ]:
from google.colab import drive

# ── Mount Drive so artifacts survive session restarts ─────────────────────────
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
OUT_GOLDEN.mkdir(parents=True, exist_ok=True)

# ── Download RVL-CDIP (skip if already cached in Drive) ───────────────────────
ARCHIVE_DRIVE = DRIVE_ROOT / "rvl-cdip.tar.gz"
ARCHIVE_URL   = "https://adamharley.com/rvl-cdip/rvl-cdip.tar.gz"

if not ARCHIVE_DRIVE.exists():
    print("Downloading RVL-CDIP (~37 GB) — grab a coffee, this takes 15-30 min...")
    !wget -q --show-progress -O "{ARCHIVE_DRIVE}" "{ARCHIVE_URL}"
    print("Download complete.")
else:
    print(f"Archive already cached: {ARCHIVE_DRIVE}")

# ── Extract to /content (fast local NVMe, not Drive) ─────────────────────────
if not DATA_ROOT.exists():
    print("Extracting archive (this takes a few minutes)...")
    !tar -xzf "{ARCHIVE_DRIVE}" -C /content/
    print("Extraction complete.")
else:
    print(f"Dataset already extracted at {DATA_ROOT}")

# ── Sanity check ──────────────────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    fpath = LABELS_DIR / f"{split}.txt"
    n = sum(1 for _ in open(fpath))
    print(f"  {split:5s}.txt : {n:>7,} samples")

## Cell 4 — Dataset class & transforms

Grayscale TIFFs → 3-channel RGB tensors with ImageNet normalization.

In [ ]:
class RvlCdipDataset(Dataset):
    """
    Reads the RVL-CDIP label files (one 'relative/path.tif int_label' per line)
    and returns (image_tensor, label_int) tuples.

    Grayscale TIFFs are expanded to 3 identical channels so the ConvNeXt
    first conv (pretrained on RGB) activates correctly.
    """

    def __init__(self, label_file: Path, images_dir: Path, transform=None):
        self.images_dir = images_dir
        self.transform  = transform
        self.samples: List[Tuple[Path, int]] = []

        with open(label_file) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rel_path, label_str = line.rsplit(" ", 1)
                self.samples.append((images_dir / rel_path, int(label_str)))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path, label = self.samples[idx]
        # Open as grayscale → convert to RGB (3 identical channels)
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


# ── ImageNet statistics — required because we use pretrained ImageNet weights ──
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ── Train transform: augmentation to improve generalization ───────────────────
train_transform = T.Compose([
    T.Resize(IMG_SIZE + 32),                          # slightly larger than crop target
    T.RandomCrop(CROP_SIZE),                           # random spatial sampling
    T.RandomHorizontalFlip(p=0.3),                     # documents can be mirrored
    T.RandomRotation(degrees=10),                      # scanner tilt
    T.ColorJitter(brightness=0.3, contrast=0.3),       # scanner exposure variance
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),   # scanner focus variation
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Val / test transform: deterministic — no random ops ──────────────────────
eval_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.CenterCrop(CROP_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# ── Build datasets ────────────────────────────────────────────────────────────
train_ds = RvlCdipDataset(LABELS_DIR / "train.txt", IMAGES_DIR, train_transform)
val_ds   = RvlCdipDataset(LABELS_DIR / "val.txt",   IMAGES_DIR, eval_transform)
test_ds  = RvlCdipDataset(LABELS_DIR / "test.txt",  IMAGES_DIR, eval_transform)

print(f"train : {len(train_ds):>7,} samples")
print(f"val   : {len(val_ds):>7,} samples")
print(f"test  : {len(test_ds):>7,} samples")

# ── Weighted sampler — ensures equal class exposure per epoch ─────────────────
# RVL-CDIP is balanced (20k/class in train), but this guards against skew.
train_labels = [s[1] for s in train_ds.samples]
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / (class_counts + 1e-6)
sample_weights = torch.tensor([class_weights[l] for l in train_labels], dtype=torch.float)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(train_ds), replacement=True)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"\nSteps per train epoch : {len(train_loader)}")
print(f"Steps per val epoch   : {len(val_loader)}")

### Visual check: sample images from each class

Verifies the dataset loaded correctly and augmentation looks sensible.

In [ ]:
# Show one raw (un-augmented) sample per class
raw_ds = RvlCdipDataset(
    LABELS_DIR / "train.txt", IMAGES_DIR,
    transform=T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])
)

# Collect first occurrence of each class
seen = {}
for img_tensor, label in raw_ds:
    if label not in seen:
        seen[label] = img_tensor
    if len(seen) == NUM_CLASSES:
        break

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("One sample per class (no augmentation)", fontsize=14, fontweight="bold")

for cls_idx, ax in enumerate(axes.flat):
    img = seen[cls_idx].permute(1, 2, 0).numpy()   # CHW → HWC
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASSES[cls_idx], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "sample_grid.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved sample_grid.png")

## Cell 5 — Build model

ConvNeXt Tiny with pretrained ImageNet weights. New 16-class head replaces the 1000-class original.

In [ ]:
def build_model() -> nn.Module:
    """
    Loads ConvNeXt Tiny with IMAGENET1K_V1 weights.
    Replaces the classifier head with Linear(768, NUM_CLASSES).
    Returns model moved to DEVICE with backbone frozen.

    Architecture:
      ConvNeXt Tiny backbone (features[0..7]) → AdaptiveAvgPool → flatten
      → LayerNorm → Linear(768, 16)
    """
    weights = models.ConvNeXt_Tiny_Weights[WEIGHTS_ENUM]
    model   = models.convnext_tiny(weights=weights)

    # The original classifier is: [LayerNorm, Flatten, Linear(768, 1000)]
    # We replace only the final Linear to keep LayerNorm pretrained.
    in_features = model.classifier[2].in_features  # 768
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

    return model.to(DEVICE)


def freeze_backbone(model: nn.Module) -> None:
    """Freeze all parameters except the classifier head."""
    for name, param in model.named_parameters():
        if "classifier" not in name:
            param.requires_grad = False


def unfreeze_last_stages(model: nn.Module) -> None:
    """
    Unfreeze ConvNeXt stages 6 and 7 (deepest feature stages).
    Stages 0-5 stay frozen — their low-level features need no adaptation.
    """
    for name, param in model.named_parameters():
        if "features.6" in name or "features.7" in name or "classifier" in name:
            param.requires_grad = True
        else:
            param.requires_grad = False


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = build_model()
freeze_backbone(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = count_trainable(model)
print(f"Total params     : {total_params:>12,}")
print(f"Trainable (probe): {trainable_params:>12,}  ({100*trainable_params/total_params:.1f}%)")
print(f"\nClassifier head  : {model.classifier}")

## Cell 6 — Training helpers (loss, optimizer, scheduler, early stopping)

In [ ]:
# ── Loss ──────────────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# ── Metrics (torchmetrics — GPU-aware) ────────────────────────────────────────
top1_metric = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=1).to(DEVICE)
top5_metric = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=5).to(DEVICE)


def make_probe_optimizer(model: nn.Module) -> torch.optim.AdamW:
    """AdamW on head parameters only."""
    head_params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(head_params, lr=PROBE_LR, weight_decay=WEIGHT_DECAY)


def make_finetune_optimizer(model: nn.Module) -> torch.optim.AdamW:
    """Differential LR: backbone stages 5× lower than head."""
    backbone_params, head_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "classifier" in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
    return torch.optim.AdamW([
        {"params": head_params,     "lr": HEAD_LR},
        {"params": backbone_params, "lr": BACKBONE_LR},
    ], weight_decay=WEIGHT_DECAY)


class EarlyStopping:
    """Halts training when validation top-1 stops improving."""

    def __init__(self, patience: int, min_delta: float = 0.0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_score = -float("inf")
        self.counter    = 0
        self.triggered  = False

    def step(self, score: float) -> bool:
        """Returns True if training should stop."""
        if score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.triggered = True
        return self.triggered


print("Training helpers ready.")

## Cell 7 — Core train / eval loop functions

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: GradScaler,
    epoch: int,
    total_epochs: int,
) -> Dict[str, float]:
    """
    One training epoch with mixed precision and gradient clipping.
    Returns dict with 'loss' and 'top1'.
    """
    model.train()
    top1_metric.reset()

    running_loss = 0.0
    steps = len(loader)
    t0 = time.time()

    for i, (images, labels) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            logits = model(images)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        top1_metric.update(logits.detach(), labels)

        # Progress print every 10% of steps
        if (i + 1) % max(1, steps // 10) == 0 or (i + 1) == steps:
            elapsed = time.time() - t0
            eta     = elapsed / (i + 1) * (steps - i - 1)
            print(
                f"  epoch {epoch:>2}/{total_epochs} "
                f"step {i+1:>4}/{steps}  "
                f"loss={running_loss/(i+1):.4f}  "
                f"top1={top1_metric.compute():.4f}  "
                f"eta={eta:.0f}s",
                end="\r"
            )

    print()  # newline after \r progress
    return {"loss": running_loss / steps, "top1": top1_metric.compute().item()}


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    temperature: float = 1.0,
) -> Dict[str, float]:
    """
    Full evaluation pass. Returns loss, top-1, top-5.
    Temperature scales the logits for calibrated softmax.
    """
    model.eval()
    top1_metric.reset()
    top5_metric.reset()

    running_loss = 0.0
    steps = len(loader)

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast():
            logits = model(images) / temperature
            loss   = criterion(logits, labels)

        running_loss += loss.item()
        top1_metric.update(logits, labels)
        top5_metric.update(logits, labels)

    return {
        "loss": running_loss / steps,
        "top1": top1_metric.compute().item(),
        "top5": top5_metric.compute().item(),
    }


print("Loop functions ready.")

## Cell 8 — Phase 1: Linear probe (frozen backbone, head-only)

Trains only the 16-class head for 5 epochs. Prevents noisy early gradients from corrupting pretrained backbone features.

In [ ]:
# History dict for plotting later
history = {
    "epoch":      [],
    "train_loss": [], "train_top1": [],
    "val_loss":   [], "val_top1":   [], "val_top5": [],
    "phase":      [],   # "probe" or "finetune"
}

probe_optimizer = make_probe_optimizer(model)
probe_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    probe_optimizer, T_max=PROBE_EPOCHS, eta_min=1e-5
)
scaler     = GradScaler()
early_stop = EarlyStopping(patience=EARLY_STOP_PAT)

print(f"=== PHASE 1: Linear probe — {PROBE_EPOCHS} epochs ===")
print(f"Trainable params: {count_trainable(model):,}  (head only)")
print()

for epoch in range(1, PROBE_EPOCHS + 1):
    train_metrics = train_one_epoch(model, train_loader, probe_optimizer, scaler, epoch, PROBE_EPOCHS)
    val_metrics   = evaluate(model, val_loader)
    probe_scheduler.step()

    history["epoch"].append(epoch)
    history["train_loss"].append(train_metrics["loss"])
    history["train_top1"].append(train_metrics["top1"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_top1"].append(val_metrics["top1"])
    history["val_top5"].append(val_metrics["top5"])
    history["phase"].append("probe")

    print(
        f"[probe epoch {epoch}/{PROBE_EPOCHS}] "
        f"train_loss={train_metrics['loss']:.4f}  "
        f"train_top1={train_metrics['top1']:.4f}  "
        f"val_top1={val_metrics['top1']:.4f}  "
        f"val_top5={val_metrics['top5']:.4f}"
    )

    if early_stop.step(val_metrics["top1"]):
        print("Early stopping triggered during probe phase.")
        break

print(f"\nProbe complete. Val top-1: {val_metrics['top1']:.4f}")

## Cell 9 — Phase 2: Partial unfreeze + fine-tune

Unfreezes backbone stages 6 & 7. Differential LR (head 5× higher than backbone).

In [ ]:
unfreeze_last_stages(model)
print(f"Trainable params after unfreeze: {count_trainable(model):,}")

ft_optimizer  = make_finetune_optimizer(model)
ft_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    ft_optimizer, T_max=FINETUNE_EPOCHS, eta_min=1e-6
)
early_stop    = EarlyStopping(patience=EARLY_STOP_PAT)
best_val_top1 = 0.0
best_ckpt_path = DRIVE_ROOT / "best_checkpoint.pt"

print(f"\n=== PHASE 2: Fine-tune — up to {FINETUNE_EPOCHS} epochs ===")
print()

for epoch in range(1, FINETUNE_EPOCHS + 1):
    global_epoch = PROBE_EPOCHS + epoch

    train_metrics = train_one_epoch(
        model, train_loader, ft_optimizer, scaler, global_epoch, TOTAL_EPOCHS
    )
    val_metrics = evaluate(model, val_loader)
    ft_scheduler.step()

    history["epoch"].append(global_epoch)
    history["train_loss"].append(train_metrics["loss"])
    history["train_top1"].append(train_metrics["top1"])
    history["val_loss"].append(val_metrics["loss"])
    history["val_top1"].append(val_metrics["top1"])
    history["val_top5"].append(val_metrics["top5"])
    history["phase"].append("finetune")

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_metrics["top1"] > best_val_top1:
        best_val_top1 = val_metrics["top1"]
        torch.save(model.state_dict(), str(best_ckpt_path))
        flag = "  ← best saved"
    else:
        flag = ""

    print(
        f"[finetune epoch {epoch}/{FINETUNE_EPOCHS}] "
        f"train_loss={train_metrics['loss']:.4f}  "
        f"train_top1={train_metrics['top1']:.4f}  "
        f"val_top1={val_metrics['top1']:.4f}  "
        f"val_top5={val_metrics['top5']:.4f}{flag}"
    )

    if early_stop.step(val_metrics["top1"]):
        print(f"Early stopping at epoch {global_epoch}.")
        break

# ── Reload best checkpoint before evaluation ──────────────────────────────────
print(f"\nLoading best checkpoint (val top-1: {best_val_top1:.4f})...")
model.load_state_dict(torch.load(str(best_ckpt_path), map_location=DEVICE))
print("Done.")

## Cell 10 — Plot training curves

Visual inspection of loss and accuracy across both training phases.

In [ ]:
epochs     = history["epoch"]
phases     = history["phase"]
probe_end  = max(i+1 for i, p in enumerate(phases) if p == "probe")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Training curves — ConvNeXt Tiny / RVL-CDIP", fontsize=14, fontweight="bold")

# ── Loss ──────────────────────────────────────────────────────────────────────
ax1.plot(epochs, history["train_loss"], label="train", color="#0077BB", linewidth=2)
ax1.plot(epochs, history["val_loss"],   label="val",   color="#EE7733", linewidth=2)
ax1.axvline(probe_end + 0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="unfreeze")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Loss")
ax1.legend(); ax1.grid(alpha=0.3)
ax1.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# ── Accuracy ──────────────────────────────────────────────────────────────────
ax2.plot(epochs, [v*100 for v in history["train_top1"]], label="train top-1", color="#0077BB", linewidth=2)
ax2.plot(epochs, [v*100 for v in history["val_top1"]],   label="val top-1",   color="#EE7733", linewidth=2)
ax2.plot(epochs, [v*100 for v in history["val_top5"]],   label="val top-5",   color="#EE7733", linewidth=2, linestyle="--")
ax2.axvline(probe_end + 0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7, label="unfreeze")
ax2.axhline(MIN_TOP1_THRESHOLD * 100, color="red", linestyle=":", linewidth=1, alpha=0.6, label="min threshold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Accuracy")
ax2.legend(); ax2.grid(alpha=0.3)
ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## Cell 11 — Temperature scaling (confidence calibration)

Fits a single scalar T on the validation set so softmax probabilities match empirical accuracy.  
T > 1 softens distributions; T < 1 sharpens them.

In [ ]:
@torch.no_grad()
def collect_logits_labels(
    model: nn.Module, loader: DataLoader
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Collect raw logits and true labels from an entire loader."""
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(images)
        all_logits.append(logits.float().cpu())
        all_labels.append(labels.cpu())
    return torch.cat(all_logits), torch.cat(all_labels)


def find_temperature(logits: torch.Tensor, labels: torch.Tensor) -> float:
    """
    Grid-search over temperatures [0.5, 4.0] to minimise NLL on val set.
    Returns the optimal scalar T.
    """
    best_T, best_nll = TEMP_INIT, float("inf")
    for T in np.arange(0.5, 4.0, 0.05):
        scaled_logits = logits / T
        nll = F.cross_entropy(scaled_logits, labels).item()
        if nll < best_nll:
            best_nll = nll
            best_T   = float(T)
    return best_T


print("Collecting val logits...")
val_logits, val_labels = collect_logits_labels(model, val_loader)

TEMPERATURE = find_temperature(val_logits, val_labels)
print(f"Optimal temperature T = {TEMPERATURE:.3f}")

# ── Show calibration effect ───────────────────────────────────────────────────
raw_conf    = F.softmax(val_logits,             dim=1).max(dim=1).values.numpy()
scaled_conf = F.softmax(val_logits / TEMPERATURE, dim=1).max(dim=1).values.numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Confidence distribution before and after temperature scaling", fontsize=13, fontweight="bold")

for ax, confs, title in [
    (axes[0], raw_conf,    f"Before (T=1.0)"),
    (axes[1], scaled_conf, f"After  (T={TEMPERATURE:.2f})"),
]:
    ax.hist(confs, bins=50, color="#0077BB", edgecolor="white", linewidth=0.3)
    ax.axvline(REVIEWER_THRESH, color="red", linestyle="--", linewidth=1.5,
               label=f"reviewer threshold ({REVIEWER_THRESH})")
    ax.set_xlabel("Max softmax confidence")
    ax.set_ylabel("Count")
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "calibration.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved calibration.png")

## Cell 12 — Full test-set evaluation

Evaluates against all 40k test images. Computes top-1, top-5, per-class accuracy.

In [ ]:
print("Collecting test logits (40k images)...")
test_logits, test_labels = collect_logits_labels(model, test_loader)

# ── Global metrics ────────────────────────────────────────────────────────────
scaled_test_logits = test_logits / TEMPERATURE

test_top1 = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=1)
test_top5 = Accuracy(task="multiclass", num_classes=NUM_CLASSES, top_k=5)
top1_val = test_top1(scaled_test_logits, test_labels).item()
top5_val = test_top5(scaled_test_logits, test_labels).item()

print(f"\nTest top-1 : {top1_val:.4f}  ({100*top1_val:.2f}%)")
print(f"Test top-5 : {top5_val:.4f}  ({100*top5_val:.2f}%)")

if top1_val < MIN_TOP1_THRESHOLD:
    print(f"\nWARNING: top-1 {top1_val:.4f} is BELOW the minimum threshold {MIN_TOP1_THRESHOLD}.")
    print("The production worker will REFUSE to start with these weights.")
    print("Train more epochs or investigate overfitting before exporting.")

# ── Per-class accuracy ────────────────────────────────────────────────────────
per_class_acc = Accuracy(
    task="multiclass", num_classes=NUM_CLASSES, average="none"
)(scaled_test_logits, test_labels).numpy()

worst_class_idx = int(np.argmin(per_class_acc))
worst_class     = CLASSES[worst_class_idx]
worst_acc       = per_class_acc[worst_class_idx]

print(f"\nPer-class test accuracy:")
for i, (cls, acc) in enumerate(zip(CLASSES, per_class_acc)):
    bar   = "█" * int(acc * 40)
    worst = " ← worst" if i == worst_class_idx else ""
    print(f"  {cls:<30s} {acc:.4f}  {bar}{worst}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
colors = ["#CC3311" if i == worst_class_idx else "#0077BB" for i in range(NUM_CLASSES)]
bars   = ax.bar(CLASSES, per_class_acc * 100, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(top1_val * 100, color="gray", linestyle="--", linewidth=1.5, label=f"mean {top1_val*100:.1f}%")
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Per-class test accuracy — ConvNeXt Tiny / RVL-CDIP", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.legend()
ax.grid(axis="y", alpha=0.3)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{acc*100:.1f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "per_class_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved per_class_accuracy.png")

## Cell 13 — Confusion matrix

Shows which classes the model confuses most — the 16×16 matrix is the most informative single diagnostic.

In [ ]:
cfm_metric = MulticlassConfusionMatrix(num_classes=NUM_CLASSES, normalize="true")
cfm = cfm_metric(scaled_test_logits, test_labels).numpy()  # shape: (16, 16)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cfm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=CLASSES, yticklabels=CLASSES,
    linewidths=0.3, linecolor="white",
    ax=ax,
    annot_kws={"size": 7},
)
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("True", fontsize=12)
ax.set_title("Normalized confusion matrix (row = true class)", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.tick_params(axis="y", rotation=0,  labelsize=8)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved confusion_matrix.png")

# ── Print top confused pairs ───────────────────────────────────────────────────
print("\nTop 5 most-confused pairs (true → predicted, off-diagonal):")
off_diag = [(cfm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES)
            for j in range(NUM_CLASSES) if i != j]
for conf_rate, true_cls, pred_cls in sorted(off_diag, reverse=True)[:5]:
    print(f"  {true_cls:<30s} → {pred_cls:<30s} {conf_rate:.3f}")

## Cell 14 — Select the 50-image golden set

- 2 highest-confidence correct predictions per class (32 'easy' images)
- 18 ambiguous cases from the top confused pairs (chosen correctly but with conf 0.65–0.80)

Golden set must be **deterministic**: eval mode, no augmentation.

In [ ]:
@torch.inference_mode()
def predict_single(
    model: nn.Module,
    img_path: Path,
    temperature: float,
) -> Dict:
    """
    Deterministic single-image inference.
    Returns class name, top-1 confidence, and all class scores.
    """
    image = Image.open(img_path).convert("RGB")
    tensor = eval_transform(image).unsqueeze(0).to(DEVICE)

    with autocast():
        logits = model(tensor) / temperature

    probs = F.softmax(logits, dim=1).squeeze().float().cpu()
    top1_conf, top1_idx = probs.max(dim=0)

    return {
        "class":      CLASSES[top1_idx.item()],
        "class_idx":  top1_idx.item(),
        "confidence": round(top1_conf.item(), 6),
        "all_scores": {CLASSES[i]: round(probs[i].item(), 6) for i in range(NUM_CLASSES)},
    }


model.eval()

# ── Collect all test samples with their predictions ───────────────────────────
print("Scoring all test samples for golden selection...")
scored_samples = []  # list of (confidence, is_correct, img_path, true_label, pred_result)

for img_path, true_label in test_ds.samples:
    result = predict_single(model, img_path, TEMPERATURE)
    scored_samples.append({
        "path":       img_path,
        "true_label": true_label,
        "true_class": CLASSES[true_label],
        "pred_class": result["class"],
        "confidence": result["confidence"],
        "correct":    result["class"] == CLASSES[true_label],
        "all_scores": result["all_scores"],
    })

# ── Strategy A: top-2 by confidence per class (correct predictions only) ──────
golden_paths = set()
golden_records = []

for cls_idx, cls_name in enumerate(CLASSES):
    class_correct = [
        s for s in scored_samples
        if s["true_label"] == cls_idx and s["correct"]
    ]
    class_correct.sort(key=lambda x: x["confidence"], reverse=True)
    for sample in class_correct[:GOLDEN_EASY]:
        golden_paths.add(sample["path"])
        golden_records.append(sample)

# ── Strategy B: ambiguous correct predictions (conf 0.65–0.80) ────────────────
ambiguous = [
    s for s in scored_samples
    if s["correct"]
    and 0.65 <= s["confidence"] <= 0.80
    and s["path"] not in golden_paths
]
# Sort by confidence ascending (most ambiguous first)
ambiguous.sort(key=lambda x: x["confidence"])

# Spread across classes — pick at most 2 per class from the ambiguous pool
class_quota: Dict[str, int] = {c: 0 for c in CLASSES}
for sample in ambiguous:
    if len(golden_records) >= GOLDEN_N:
        break
    if class_quota[sample["true_class"]] >= 2:
        continue
    golden_paths.add(sample["path"])
    golden_records.append(sample)
    class_quota[sample["true_class"]] += 1

print(f"\nGolden set: {len(golden_records)} images selected")
print(f"  Easy (top-2 per class) : {GOLDEN_EASY * NUM_CLASSES}")
print(f"  Ambiguous (0.65-0.80)  : {len(golden_records) - GOLDEN_EASY * NUM_CLASSES}")

## Cell 15 — Export golden set files

Copies the 50 TIFFs to Drive and writes `golden_expected.json`.

In [ ]:
OUT_GOLDEN.mkdir(parents=True, exist_ok=True)

golden_expected = {}

for record in golden_records:
    src_path  = record["path"]
    filename  = src_path.name
    dst_path  = OUT_GOLDEN / filename

    # Ensure unique filenames (different directories may share names)
    if dst_path.exists():
        stem   = src_path.stem
        suffix = src_path.suffix
        unique = src_path.parent.name
        filename = f"{unique}_{stem}{suffix}"
        dst_path = OUT_GOLDEN / filename

    shutil.copy2(src_path, dst_path)

    golden_expected[filename] = {
        "class":      record["pred_class"],
        "confidence": record["confidence"],
        "all_scores": record["all_scores"],
    }

# ── Write golden_expected.json ────────────────────────────────────────────────
with open(OUT_EXPECTED, "w") as f:
    json.dump(golden_expected, f, indent=2)

print(f"Exported {len(golden_expected)} golden records.")
print(f"  TIFFs  → {OUT_GOLDEN}")
print(f"  JSON   → {OUT_EXPECTED}")

# ── Visual preview of golden set ──────────────────────────────────────────────
n_preview = min(16, len(golden_records))
fig, axes = plt.subplots(2, 8, figsize=(22, 6))
fig.suptitle("Golden set — first 16 images", fontsize=13, fontweight="bold")

for ax, record in zip(axes.flat, golden_records[:n_preview]):
    img = Image.open(record["path"]).convert("RGB")
    img.thumbnail((224, 224))
    ax.imshow(img, cmap="gray")
    ax.set_title(
        f"{record['true_class']}\n{record['confidence']:.2f}",
        fontsize=7, color="green" if record["correct"] else "red"
    )
    ax.axis("off")

plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / "golden_preview.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Saved golden_preview.png")

## Cell 16 — Export weights & model card

Saves `classifier.pt`, computes SHA-256, writes `model_card.json`.

In [ ]:
import platform

# ── Save weights ──────────────────────────────────────────────────────────────
# Always save state_dict only — safer to load across PyTorch versions.
torch.save(model.state_dict(), str(OUT_WEIGHTS))
print(f"Saved weights → {OUT_WEIGHTS}")
print(f"File size     : {OUT_WEIGHTS.stat().st_size / 1e6:.1f} MB")

# ── SHA-256 ───────────────────────────────────────────────────────────────────
sha256 = hashlib.sha256()
with open(OUT_WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
WEIGHTS_SHA256 = sha256.hexdigest()
print(f"SHA-256       : {WEIGHTS_SHA256}")

# ── Model card ────────────────────────────────────────────────────────────────
model_card = {
    "schema_version":    "1.0",
    "backbone":          BACKBONE,
    "weights_enum":      WEIGHTS_ENUM,
    "num_classes":       NUM_CLASSES,
    "classes":           CLASSES,
    "image_size":        IMG_SIZE,
    "imagenet_mean":     IMAGENET_MEAN,
    "imagenet_std":      IMAGENET_STD,
    "temperature":       TEMPERATURE,
    "sha256":            WEIGHTS_SHA256,
    "freeze_policy":     "linear_probe_then_partial_unfreeze",
    "probe_epochs":      PROBE_EPOCHS,
    "finetune_epochs":   FINETUNE_EPOCHS,
    "label_smoothing":   LABEL_SMOOTHING,
    "metrics": {
        "test_top1":     round(top1_val, 6),
        "test_top5":     round(top5_val, 6),
        "worst_class":   worst_class,
        "worst_class_acc": round(float(worst_acc), 6),
        "per_class_acc": {
            CLASSES[i]: round(float(per_class_acc[i]), 6)
            for i in range(NUM_CLASSES)
        },
    },
    "min_top1_threshold": MIN_TOP1_THRESHOLD,
    "reviewer_threshold": REVIEWER_THRESH,
    "environment": {
        "python":        platform.python_version(),
        "torch":         torch.__version__,
        "torchvision":   __import__("torchvision").__version__,
        "cuda":          torch.version.cuda,
        "gpu":           torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "seed":          SEED,
    },
}

with open(OUT_CARD, "w") as f:
    json.dump(model_card, f, indent=2)

print(f"\nModel card    → {OUT_CARD}")
print(json.dumps({k: v for k, v in model_card.items() if k != "classes"}, indent=2))

## Cell 17 — Golden set replay verification

Reloads weights from disk and re-runs all 50 golden images.  
**Every output must match `golden_expected.json` within 1e-6 or the export is invalid.**

In [ ]:
# ── Reload model from disk ────────────────────────────────────────────────────
verify_model = build_model()

# Verify SHA-256 before loading (mirrors production load_model())
sha256 = hashlib.sha256()
with open(OUT_WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
loaded_sha = sha256.hexdigest()
assert loaded_sha == WEIGHTS_SHA256, (
    f"SHA-256 mismatch!  expected {WEIGHTS_SHA256}  got {loaded_sha}"
)
print(f"SHA-256 verified: {loaded_sha[:16]}...")

verify_model.load_state_dict(
    torch.load(str(OUT_WEIGHTS), map_location=DEVICE)
)
verify_model.eval()

# ── Load expected outputs ─────────────────────────────────────────────────────
with open(OUT_EXPECTED) as f:
    expected = json.load(f)

# ── Run all 50 images ─────────────────────────────────────────────────────────
TOLERANCE = 1e-6
passed, failed, failures = 0, 0, []

for filename, exp in expected.items():
    img_path = OUT_GOLDEN / filename
    result   = predict_single(verify_model, img_path, TEMPERATURE)

    class_ok = result["class"] == exp["class"]
    conf_ok  = abs(result["confidence"] - exp["confidence"]) < TOLERANCE

    if class_ok and conf_ok:
        passed += 1
    else:
        failed += 1
        failures.append(
            f"  FAIL {filename}\n"
            f"    class     : expected={exp['class']}  got={result['class']}\n"
            f"    confidence: expected={exp['confidence']:.6f}  got={result['confidence']:.6f}"
        )

print(f"\nGolden replay: {passed}/{passed+failed} passed")
if failures:
    print("\nFAILURES:")
    for f in failures:
        print(f)
    raise AssertionError(
        f"{failed} golden images failed — artifacts are invalid. "
        "Do NOT commit these weights."
    )
else:
    print("All golden tests passed. Artifacts are ready to commit.")
    print(f"\n{'='*60}")
    print(f"  SUBMIT this in your repo README:")
    print(f"  Backbone       : {BACKBONE} / {WEIGHTS_ENUM}")
    print(f"  Freeze policy  : linear probe → partial unfreeze")
    print(f"  Test top-1     : {top1_val:.4f}  ({100*top1_val:.2f}%)")
    print(f"  Test top-5     : {top5_val:.4f}  ({100*top5_val:.2f}%)")
    print(f"  Worst class    : {worst_class} ({100*worst_acc:.2f}%)")
    print(f"  Temperature T  : {TEMPERATURE:.3f}")
    print(f"  SHA-256        : {WEIGHTS_SHA256}")
    print(f"{'='*60}")

## Cell 18 — Production `model.py` (copy this into `app/classifier/model.py`)

This is the exact code the inference worker imports. It includes:  
- SHA-256 check at startup  
- Minimum accuracy threshold check  
- Temperature-calibrated `predict()` returning `(class_name, confidence, all_scores)`

In [ ]:
PRODUCTION_MODEL_PY = '''
"""
app/classifier/model.py

Loads classifier.pt and exposes predict().
Raises RuntimeError at startup if weights are missing, SHA-256 mismatches,
or test top-1 is below the threshold in model_card.json.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
import io

MODELS_DIR   = Path(__file__).parent / "models"
WEIGHTS_PATH = MODELS_DIR / "classifier.pt"
CARD_PATH    = MODELS_DIR / "model_card.json"


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()


def load_model() -> Tuple[nn.Module, dict]:
    """
    Load and validate classifier weights.

    Returns:
        model  - ConvNeXt with loaded weights, in eval mode, on CPU
        card   - parsed model_card.json

    Raises:
        RuntimeError if weights are missing, SHA-256 mismatches,
        or test top-1 is below the threshold.
    """
    if not WEIGHTS_PATH.exists():
        raise RuntimeError(f"Classifier weights not found: {WEIGHTS_PATH}")
    if not CARD_PATH.exists():
        raise RuntimeError(f"Model card not found: {CARD_PATH}")

    with open(CARD_PATH) as f:
        card = json.load(f)

    # ── SHA-256 integrity check ───────────────────────────────────────────────
    actual_sha = _sha256(WEIGHTS_PATH)
    if actual_sha != card["sha256"]:
        raise RuntimeError(
            f"SHA-256 mismatch for {WEIGHTS_PATH}\n"
            f"  expected : {card[\"sha256\"]}\n"
            f"  actual   : {actual_sha}"
        )

    # ── Accuracy gate ─────────────────────────────────────────────────────────
    top1 = card["metrics"]["test_top1"]
    threshold = card["min_top1_threshold"]
    if top1 < threshold:
        raise RuntimeError(
            f"Model top-1 {top1:.4f} is below minimum threshold {threshold}."
        )

    # ── Build and load model ──────────────────────────────────────────────────
    num_classes = card["num_classes"]
    weights     = models.ConvNeXt_Tiny_Weights[card["weights_enum"]]
    model       = models.convnext_tiny(weights=None)  # no re-download
    model.classifier[2] = nn.Linear(
        model.classifier[2].in_features, num_classes
    )
    model.load_state_dict(
        torch.load(str(WEIGHTS_PATH), map_location="cpu")
    )
    model.eval()

    return model, card


def _make_transform(card: dict) -> T.Compose:
    size = card["image_size"]
    return T.Compose([
        T.Resize(size),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize(mean=card["imagenet_mean"], std=card["imagenet_std"]),
    ])


@torch.inference_mode()
def predict(
    model: nn.Module,
    image_bytes: bytes,
    card: dict,
) -> Tuple[str, float, Dict[str, float]]:
    """
    Classify a document image.

    Args:
        model       - loaded model from load_model()
        image_bytes - raw bytes of a TIFF, PNG, or JPEG
        card        - model card dict from load_model()

    Returns:
        (class_name, confidence, all_scores)
        confidence and all_scores are temperature-calibrated probabilities.
    """
    image     = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    transform = _make_transform(card)
    tensor    = transform(image).unsqueeze(0)

    logits = model(tensor) / card["temperature"]
    probs  = F.softmax(logits, dim=1).squeeze()

    top1_conf, top1_idx = probs.max(dim=0)
    classes     = card["classes"]
    pred_class  = classes[top1_idx.item()]
    confidence  = round(top1_conf.item(), 6)
    all_scores  = {classes[i]: round(probs[i].item(), 6) for i in range(len(classes))}

    return pred_class, confidence, all_scores
'''

# Write to Drive so you can copy it into the repo
out_path = DRIVE_ROOT / "model.py"
with open(out_path, "w") as f:
    f.write(PRODUCTION_MODEL_PY.strip())

print(f"Production model.py written to {out_path}")
print()
print("Next steps:")
print("  1. cp classifier.pt   → app/classifier/models/classifier.pt")
print("  2. cp model_card.json → app/classifier/models/model_card.json")
print("  3. cp golden_images/  → app/classifier/eval/golden_images/")
print("  4. cp golden_expected.json → app/classifier/eval/golden_expected.json")
print("  5. cp model.py        → app/classifier/model.py")
print("  6. git lfs track '*.pt' && git add . && git commit")

## Summary of artifacts produced

| File | What it is |
|---|---|
| `classifier.pt` | ConvNeXt Tiny state_dict — 110 MB, tracked with git LFS |
| `model_card.json` | SHA-256, metrics, temperature, environment fingerprint |
| `golden_images/` | 50 TIFFs (32 easy + 18 ambiguous) |
| `golden_expected.json` | Expected class + confidence per image, tolerance 1e-6 |
| `model.py` | Drop-in production inference module |
| `training_curves.png` | Loss + accuracy over epochs |
| `per_class_accuracy.png` | Bar chart of per-class test accuracy |
| `confusion_matrix.png` | 16×16 normalized confusion matrix |
| `calibration.png` | Confidence histogram before/after temperature scaling |
| `sample_grid.png` | One sample per class (visual sanity check) |
| `golden_preview.png` | Thumbnail grid of the 50 golden images |